# 00_full_sweep — single-GPU clean experiment orchestrator

Run this notebook when only one Kaggle GPU is available. It executes the high-value experiments sequentially, reuses feature caches, frees GPU memory between optional encoders, and writes a combined leaderboard. It never uses train/test overlap information, test labels, or Champion 01 logic.

In [ ]:
# Bootstrap the exact clean-suite code from the locked public GitHub commit.
import subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/nikhilwankhedee/pareidolia-paradox.git'
REPO_REF = 'f0153a4e40d3327b2fbf6b6296217162e841b9a5'
REPO_ROOT = Path('/kaggle/working/pareidolia-paradox')
if not (REPO_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--depth=1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', REPO_REF], check=True)
SUITE = REPO_ROOT / 'experiments' / 'clean_breakthrough'
assert SUITE.is_dir(), f'Clean suite missing from {REPO_REF}'
sys.path.insert(0, str(SUITE))
print('fetched repository:', REPO_ROOT)
print('checked out commit:', REPO_REF)
print('suite:', SUITE)


In [ ]:
from pathlib import Path
import gc, json, sys, traceback
import pandas as pd
from src.dataset import discover_dataset, add_hashes
from src.validation import assign_folds, assert_no_overlap
from src.pipeline import run_experiment
from src.utils import seed_all
seed_all(42)
OUTPUT = Path('/kaggle/working/clean_breakthrough')
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA = discover_dataset()
TRAIN = assign_folds(add_hashes(DATA['train'], DATA['train_images'], verify=True), n_splits=5, seed=42, hash_col='hash')
assert_no_overlap(TRAIN)
print('dataset:', DATA['root'])
print('train/test:', len(TRAIN), len(DATA['test']))
print('output/cache:', OUTPUT)


In [ ]:
# Ordered for one GPU: CPU baselines first, then foundation encoders, then hybrids.
EXPERIMENTS = [
    'morphology', 'sfs',
    'dino_b14', 'dino_b14_azimuth', 'dino_multiview',
    'clip', 'pretrained_vision',
    'dino_morphology', 'dino_sfs', 'tta'
]
records = []
for kind in EXPERIMENTS:
    print('\n' + '=' * 72 + '\nRUNNING ' + kind + '\n' + '=' * 72)
    try:
        result = run_experiment(DATA, TRAIN, DATA['test'], kind, OUTPUT, folds=5, seed=42)
        row = dict(result['metrics'])
        row.update({'experiment_id': kind, 'artifact_path': result['cache']})
        records.append(row)
        print('OOF metrics:', row)
    except (RuntimeError, ImportError, OSError) as exc:
        print('SKIPPED:', kind, repr(exc))
        records.append({'experiment_id': kind, 'status': 'skipped', 'error': repr(exc)})
    finally:
        gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass
(OUTPUT / 'results').mkdir(parents=True, exist_ok=True)
leaderboard = pd.DataFrame(records)
leaderboard.to_csv(OUTPUT / 'results' / 'leaderboard.csv', index=False)
leaderboard


Review `results/leaderboard.csv` and the per-experiment OOF files before considering any model for promotion. A skipped optional phase is expected if Kaggle cannot download its encoder. No submission is created by this notebook.